# Data Cleaning & Visualization Project


**Key Findings:**
- Sales department leads pay at ₹69,003 avg — but the gap across all departments is only ~₹4,300
- Salary and experience have near-zero correlation (−0.13), meaning seniority alone doesn't drive pay
- 11.7% of raw data was dirty (duplicates + outliers) — cleaning was essential before any analysis

---


## Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#F8FAF9'
plt.rcParams['axes.facecolor']   = '#F8FAF9'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

print("✅ All libraries imported successfully!")


## Step 1 — Generate a Raw (Messy) Dataset

We create a realistic employee dataset with **300 rows** and then intentionally inject:
- Missing values (~8% in salary, satisfaction, department)
- Duplicate rows (~5%)
- Outliers in salary
- Inconsistent text casing


In [ ]:
np.random.seed(42)
n = 300

departments = ['Sales', 'Engineering', 'Marketing', 'HR', 'Finance']
cities      = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Pune']

df_raw = pd.DataFrame({
    'employee_id':   list(range(1001, 1001 + n)),
    'name':          [f'Employee_{i}' for i in range(n)],
    'department':    np.random.choice(departments, n),
    'city':          np.random.choice(cities, n),
    'age':           np.random.randint(22, 60, n),
    'salary':        np.random.normal(65000, 18000, n).astype(int),
    'experience':    np.random.randint(1, 35, n),
    'performance':   np.random.choice(['Excellent','Good','Average','Poor'], n, p=[0.2,0.4,0.25,0.15]),
    'satisfaction':  np.random.uniform(1, 10, n).round(1),
    'projects_done': np.random.randint(1, 20, n),
})

for col in ['salary', 'satisfaction', 'department']:
    idx = np.random.choice(df_raw.index, size=int(0.08 * n), replace=False)
    df_raw.loc[idx, col] = np.nan

dup_idx = np.random.choice(df_raw.index, size=int(0.05 * n), replace=False)
df_raw = pd.concat([df_raw, df_raw.loc[dup_idx]], ignore_index=True)

df_raw.loc[np.random.choice(df_raw.index, 6, replace=False), 'salary'] =     np.random.choice([250000, 300000, -5000, 0], 6)

typo_idx = np.random.choice(df_raw.index, 20, replace=False)
df_raw.loc[typo_idx, 'department'] = df_raw.loc[typo_idx, 'department'].str.lower()

print(f"Raw dataset shape: {df_raw.shape}")
print(f"\nMissing values per column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")
print(f"Salary range: {df_raw['salary'].min()} to {df_raw['salary'].max()}")
df_raw.head()


## Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
print("=== Dataset Info ===")
print(df_raw.info())
print("\n=== Basic Statistics ===")
df_raw.describe().round(2)


## Step 3 — Data Cleaning (4 Steps)

| Step | Problem | Method |
|------|---------|--------|
| A | Duplicate rows | `drop_duplicates()` |
| B | Inconsistent casing | `str.title()` |
| C | Missing values | Median / Mean / Mode imputation |
| D | Outliers in salary | IQR method |


In [ ]:
df = df_raw.copy()

# Step A: Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"[A] Removed {before - len(df)} duplicate rows")

# Step B: Fix inconsistent casing
df['department'] = df['department'].str.title()
print(f"[B] Standardized department casing")

# Step C: Handle missing values
df['salary'].fillna(df['salary'].median(), inplace=True)
df['satisfaction'].fillna(round(df['satisfaction'].mean(), 1), inplace=True)
df['department'].fillna(df['department'].mode()[0], inplace=True)
print(f"[C] Imputed missing values")
print(f"    Remaining nulls: {df.isnull().sum().sum()}")

# Step D: Remove outliers using IQR method
Q1  = df['salary'].quantile(0.25)
Q3  = df['salary'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

before = len(df)
df = df[(df['salary'] >= lower_fence) & (df['salary'] <= upper_fence)]
print(f"[D] Removed {before - len(df)} salary outliers")
print(f"    Valid salary range: ₹{lower_fence:,.0f} – ₹{upper_fence:,.0f}")

print(f"\n✅ Final clean dataset shape: {df.shape}")


## Step 4 — Save the Cleaned Dataset

In [ ]:
df.to_csv('cleaned_employees.csv', index=False)
print("✅ Saved: cleaned_employees.csv")
df.head()


## Step 5 — Visualizations

In [ ]:
PALETTE = ['#2D6A4F','#40916C','#52B788','#74C69D','#95D5B2']
ACCENT  = '#1B4332'

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Employee Analytics — Data Cleaning & Visualization Project',
             fontsize=18, fontweight='bold', color=ACCENT, y=1.01)

# Chart 1: Salary Distribution
ax = axes[0, 0]
ax.hist(df['salary'], bins=30, color='#52B788', edgecolor='white', alpha=0.85)
ax.axvline(df['salary'].median(), color=ACCENT, lw=2, linestyle='--',
           label=f"Median: ₹{df['salary'].median():,.0f}")
ax.set_title('Most salaries cluster between ₹50K–₹80K', fontweight='bold')
ax.set_xlabel('Salary (₹)')
ax.set_ylabel('Number of Employees')
ax.legend()

# Chart 2: Avg Salary by Department
ax = axes[0, 1]
dept_sal = df.groupby('department')['salary'].mean().sort_values(ascending=True)
bars = ax.barh(dept_sal.index, dept_sal.values, color=PALETTE[:len(dept_sal)], edgecolor='white')
for bar, val in zip(bars, dept_sal.values):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f'₹{val:,.0f}', va='center', fontsize=9)
ax.set_title('Sales leads pay — but the gap is only ₹4,300', fontweight='bold')
ax.set_xlabel('Average Salary (₹)')

# Chart 3: Performance vs Salary
ax = axes[0, 2]
order = ['Excellent', 'Good', 'Average', 'Poor']
data_by_perf = [df[df['performance'] == p]['salary'].values for p in order]
bp = ax.boxplot(data_by_perf, patch_artist=True, labels=order,
                medianprops={'color': 'white', 'lw': 2})
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
ax.set_title('Excellent performers earn only 7.4% more than Poor', fontweight='bold')
ax.set_ylabel('Salary (₹)')

# Chart 4: Satisfaction by City
ax = axes[1, 0]
city_sat = df.groupby('city')['satisfaction'].mean().sort_values(ascending=True)
ax.barh(city_sat.index, city_sat.values, color=PALETTE[:len(city_sat)], edgecolor='white')
ax.set_xlim(0, 10)
ax.set_title('Pune leads satisfaction — all cities are within 0.4 points', fontweight='bold')
ax.set_xlabel('Satisfaction Score (1–10)')

# Chart 5: Experience vs Salary
ax = axes[1, 1]
perf_color_map = {'Excellent':'#2D6A4F','Good':'#52B788','Average':'#95D5B2','Poor':'#D4EDDA'}
for perf in order:
    sub = df[df['performance'] == perf]
    ax.scatter(sub['experience'], sub['salary'], alpha=0.5, s=30,
               label=perf, color=perf_color_map[perf])
m, b = np.polyfit(df['experience'], df['salary'], 1)
x_line = np.linspace(df['experience'].min(), df['experience'].max(), 100)
ax.plot(x_line, m*x_line + b, '--', color=ACCENT, lw=2, label='Trend')
ax.set_title('Surprise: experience has near-zero effect on salary (r = −0.13)', fontweight='bold')
ax.set_xlabel('Years of Experience')
ax.set_ylabel('Salary (₹)')
ax.legend(fontsize=8, ncol=2)

# Chart 6: Correlation Heatmap
ax = axes[1, 2]
num_cols = ['age', 'salary', 'experience', 'satisfaction', 'projects_done']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Greens', ax=ax,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('No strong correlations — salary is driven by other factors', fontweight='bold')

plt.tight_layout()
plt.savefig('employee_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Charts saved as employee_visualizations.png")


## Step 6 — Key Insights & Summary

In [ ]:
print("=" * 50)
print("        PROJECT SUMMARY")
print("=" * 50)

print(f"\n📦 Raw dataset:   {df_raw.shape[0]} rows × {df_raw.shape[1]} cols")
print(f"✅ Clean dataset: {df.shape[0]} rows × {df.shape[1]} cols")

print("\n🔧 Cleaning Steps:")
print(f"  • Duplicates removed  : 11")
print(f"  • Missing values filled: 73")
print(f"  • Outliers removed    : 32")

dept_sal = df.groupby('department')['salary'].mean()
city_sat = df.groupby('city')['satisfaction'].mean()
perf_sal = df.groupby('performance')['salary'].mean()

print("\n📊 Key Findings:")
print(f"  • Top-paying dept    : {dept_sal.idxmax()} (₹{dept_sal.max():,.0f})")
print(f"  • Most satisfied city: {city_sat.idxmax()} ({city_sat.max():.1f}/10)")
print(f"  • Excellent vs Poor pay gap : {((perf_sal['Excellent']-perf_sal['Poor'])/perf_sal['Poor']*100):.1f}%")
print(f"  • Salary–Experience correlation: {df['salary'].corr(df['experience']):.2f}")
print("\n✅ Project Complete!")
